# Formalisation des 3 Analyses Minimales

**Sprint 3**

Ce notebook formalise les **3 analyses imposées** par le sujet :

| # | Analyse | Pipeline | Table/Vue résultat |
|---|---------|----------|--------------------|
| 1 | **Segmentation clients/producteurs** | K-Means clustering (Sprint 2) | `v_current_customer_segments`, `v_current_producer_segments` |
| 2 | **Détection d'anomalies** | Isolation Forest (Sprint 3) | `v_current_anomalies` |
| 3 | **Analyse comportementale** | Métriques de vente agrégées | `v_sales_summary` |

Chaque analyse est connectée à la base de données et produit des visualisations exploitables.

In [3]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

from data.pipeline.db import get_engine, query_df

engine = get_engine()
print("Connexion établie.")

Connexion établie.


---
## Analyse 1 — Segmentation (Clustering K-Means)

**Objectif** : Regrouper les clients et producteurs en segments homogènes pour adapter les actions marketing et commerciales.

**Méthode** : K-Means avec recherche automatique du K optimal via le silhouette score. Features RFM (Recency, Frequency, Monetary) pour les clients, features d'activité pour les producteurs.

**Tables** : `customer_segments`, `producer_segments`, `clustering_runs`

In [4]:
# --- Segmentation Clients ---
df_customers = query_df("SELECT * FROM v_current_customer_segments", engine)
print(f"Nombre de clients segmentés : {len(df_customers)}")
print(f"\nDistribution des segments clients :")
print(df_customers["cluster_label"].value_counts())

Nombre de clients segmentés : 1000

Distribution des segments clients :
cluster_label
Fideles premium           408
Acheteurs occasionnels    387
Clients dormants          205
Name: count, dtype: int64


In [5]:
# Visualisation segments clients
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des segments
df_customers["cluster_label"].value_counts().plot(
    kind="bar", ax=axes[0], color="steelblue", edgecolor="black"
)
axes[0].set_title("Distribution des segments clients")
axes[0].set_xlabel("Segment")
axes[0].set_ylabel("Nombre de clients")
axes[0].tick_params(axis="x", rotation=45)

# RFM moyen par segment
rfm_means = df_customers.groupby("cluster_label")[["recency_days", "frequency", "monetary"]].mean()
rfm_means.plot(kind="bar", ax=axes[1], edgecolor="black")
axes[1].set_title("Profil RFM moyen par segment")
axes[1].set_xlabel("Segment")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(title="Feature")

plt.tight_layout()
plt.savefig("../outputs/analyse1_segmentation_clients.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : ../outputs/analyse1_segmentation_clients.png")

Figure sauvegardée : ../outputs/analyse1_segmentation_clients.png


/tmp/ipykernel_236406/2358128291.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# --- Segmentation Producteurs ---
df_producers = query_df("SELECT * FROM v_current_producer_segments", engine)
print(f"Nombre de producteurs segmentés : {len(df_producers)}")
print(f"\nDistribution des segments producteurs :")
print(df_producers["cluster_label"].value_counts())

Nombre de producteurs segmentés : 200

Distribution des segments producteurs :
cluster_label
Nouveau producteur            115
Gros producteur diversifie     85
Name: count, dtype: int64


In [7]:
# Visualisation segments producteurs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_producers["cluster_label"].value_counts().plot(
    kind="bar", ax=axes[0], color="forestgreen", edgecolor="black"
)
axes[0].set_title("Distribution des segments producteurs")
axes[0].set_xlabel("Segment")
axes[0].set_ylabel("Nombre de producteurs")
axes[0].tick_params(axis="x", rotation=45)

prod_means = df_producers.groupby("cluster_label")[["total_revenue", "n_orders_received", "n_products"]].mean()
prod_means.plot(kind="bar", ax=axes[1], edgecolor="black")
axes[1].set_title("Profil moyen par segment producteur")
axes[1].set_xlabel("Segment")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(title="Feature")

plt.tight_layout()
plt.savefig("../outputs/analyse1_segmentation_producteurs.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : ../outputs/analyse1_segmentation_producteurs.png")

Figure sauvegardée : ../outputs/analyse1_segmentation_producteurs.png


/tmp/ipykernel_236406/2847596909.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Métriques de qualité du clustering
df_runs = query_df("""
    SELECT run_type, n_clusters, silhouette_score, inertia, finished_at
    FROM clustering_runs
    WHERE status = 'success'
    ORDER BY finished_at DESC
    LIMIT 2
""", engine)
print("Dernières exécutions de clustering :")
print(df_runs.to_string(index=False))

Dernières exécutions de clustering :
run_type  n_clusters  silhouette_score  inertia                      finished_at
producer           2            0.2912   781.97 2026-03-23 16:29:47.735954+00:00
customer           3            0.2124  3972.29 2026-03-23 16:29:47.586858+00:00


---
## Analyse 2 — Détection d'anomalies (Isolation Forest)

**Objectif** : Identifier les transactions suspectes (paiements échoués non simulés, montants inhabituels, fréquences anormales) pour alimenter un système d'alerte.

**Méthode** : Isolation Forest sur 8 features transactionnelles normalisées, avec contamination à ~8%. Labeling automatique des types d'anomalies.

**Tables** : `anomalies`, `anomaly_runs`

In [9]:
df_anomalies = query_df("SELECT * FROM v_current_anomalies", engine)
print(f"Nombre d'anomalies détectées : {len(df_anomalies)}")
print(f"\nDistribution par type :")
print(df_anomalies["anomaly_type"].value_counts())

Nombre d'anomalies détectées : 382

Distribution par type :
anomaly_type
Anomalie non classifiee       299
Montant anormalement eleve     52
Montant anormalement bas       19
Erreur simulee                 12
Name: count, dtype: int64


In [10]:
# Visualisation des anomalies
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution par type
df_anomalies["anomaly_type"].value_counts().plot(
    kind="barh", ax=axes[0], color="crimson", edgecolor="black"
)
axes[0].set_title("Types d'anomalies détectées")
axes[0].set_xlabel("Nombre")

# Score d'anomalie par type
df_anomalies.boxplot(column="anomaly_score", by="anomaly_type", ax=axes[1], rot=45)
axes[1].set_title("Score d'anomalie par type")
axes[1].set_xlabel("Type")
axes[1].set_ylabel("Score")
plt.suptitle("")  # Remove auto-title from boxplot

# Montant des transactions anormales vs normales
df_all_orders = query_df("""
    SELECT total_amount FROM orders WHERE status != 'draft'
""", engine)
axes[2].hist(df_all_orders["total_amount"], bins=50, alpha=0.5, label="Toutes", color="steelblue")
axes[2].hist(df_anomalies["order_amount"], bins=30, alpha=0.7, label="Anomalies", color="crimson")
axes[2].set_title("Distribution des montants")
axes[2].set_xlabel("Montant (EUR)")
axes[2].set_ylabel("Fréquence")
axes[2].legend()

plt.tight_layout()
plt.savefig("../outputs/analyse2_anomalies.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : ../outputs/analyse2_anomalies.png")

Figure sauvegardée : ../outputs/analyse2_anomalies.png


/tmp/ipykernel_236406/4244820499.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Métriques du run d'anomalies
df_anom_runs = query_df("""
    SELECT algorithm, n_anomalies, contamination, parameters, finished_at
    FROM anomaly_runs
    WHERE status = 'success'
    ORDER BY finished_at DESC
    LIMIT 1
""", engine)
print("Dernière exécution de détection d'anomalies :")
print(df_anom_runs.to_string(index=False))

Dernière exécution de détection d'anomalies :
      algorithm  n_anomalies  contamination                                                                                                                                                                                                                                                                                                             parameters                      finished_at
IsolationForest          382           0.08 {'scaler': 'StandardScaler', 'features': ['order_amount', 'n_items', 'avg_item_price', 'hour_of_day', 'day_of_week', 'days_since_last_order', 'client_avg_basket', 'amount_vs_avg_ratio'], 'algorithm': 'IsolationForest', 'n_estimators': 200, 'random_state': 42, 'contamination': 0.08, 'total_transactions': 4765} 2026-03-30 06:23:40.732346+00:00


---
## Analyse 3 — Comportement d'achat (Métriques de vente)

**Objectif** : Analyser les comportements d'achat : saisonnalité, catégories populaires, distribution géographique, paniers moyens.

**Méthode** : Agrégation SQL via la vue `v_sales_summary` (pré-calculée) + analyses temporelles et catégorielles.

**Tables** : `v_sales_summary`, `orders`, `order_items`, `products`

In [12]:
# Métriques globales
df_sales = query_df("SELECT * FROM v_sales_summary", engine)
print(f"Jours de vente analysés : {df_sales['sale_day'].nunique()}")
print(f"Revenu total : {df_sales['revenue'].sum():,.2f} EUR")
print(f"Nombre total de commandes : {df_sales['nb_orders'].sum()}")
print(f"Panier moyen global : {df_sales['avg_basket'].mean():,.2f} EUR")

Jours de vente analysés : 181
Revenu total : 587,563.27 EUR
Nombre total de commandes : 11994
Panier moyen global : 204.47 EUR


In [13]:
# Ventes par catégorie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_revenue = df_sales.groupby("category")["revenue"].sum().sort_values(ascending=True)
cat_revenue.plot(kind="barh", ax=axes[0], color="darkorange", edgecolor="black")
axes[0].set_title("Revenu par catégorie de produit")
axes[0].set_xlabel("Revenu (EUR)")

cat_orders = df_sales.groupby("category")["nb_orders"].sum().sort_values(ascending=True)
cat_orders.plot(kind="barh", ax=axes[1], color="teal", edgecolor="black")
axes[1].set_title("Nombre de commandes par catégorie")
axes[1].set_xlabel("Commandes")

plt.tight_layout()
plt.savefig("../outputs/analyse3_categories.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : ../outputs/analyse3_categories.png")

Figure sauvegardée : ../outputs/analyse3_categories.png


/tmp/ipykernel_236406/2219452540.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# Évolution temporelle des ventes
daily_sales = df_sales.groupby("sale_day").agg(
    revenue=("revenue", "sum"),
    nb_orders=("nb_orders", "sum"),
).sort_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(daily_sales.index, daily_sales["revenue"], color="steelblue", linewidth=0.8)
axes[0].fill_between(daily_sales.index, daily_sales["revenue"], alpha=0.3, color="steelblue")
axes[0].set_title("Revenu journalier")
axes[0].set_ylabel("Revenu (EUR)")

axes[1].bar(daily_sales.index, daily_sales["nb_orders"], color="darkorange", width=1.0)
axes[1].set_title("Commandes journalières")
axes[1].set_ylabel("Nombre de commandes")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.savefig("../outputs/analyse3_temporel.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : ../outputs/analyse3_temporel.png")

Figure sauvegardée : ../outputs/analyse3_temporel.png


/tmp/ipykernel_236406/499397528.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Ventes par région
region_revenue = df_sales.groupby("location_region")["revenue"].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
region_revenue.tail(15).plot(kind="barh", ax=ax, color="mediumpurple", edgecolor="black")
ax.set_title("Top 15 régions par revenu")
ax.set_xlabel("Revenu (EUR)")
ax.set_ylabel("Région")

plt.tight_layout()
plt.savefig("../outputs/analyse3_regions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : ../outputs/analyse3_regions.png")

Figure sauvegardée : ../outputs/analyse3_regions.png


/tmp/ipykernel_236406/3356272892.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Synthèse

Les 3 analyses minimales requises sont implémentées et exploitables :

| Analyse | Statut | Pipeline | Endpoint API | Vue SQL |
|---------|--------|----------|--------------|---------|
| Segmentation | Opérationnel | `run_pipeline.py --customers --producers` | `GET /data/clustering/customers` | `v_current_customer_segments`, `v_current_producer_segments` |
| Anomalies | Opérationnel | `run_pipeline.py --anomalies` | `GET /data/anomalies` | `v_current_anomalies` |
| Comportement | Opérationnel | Vue SQL pré-agrégée | `GET /data/sales-metrics` | `v_sales_summary` |

**Pipeline complet** : `python -m data.pipeline.run_pipeline` (exécute les 3 analyses)

**Accès données** :
- API REST : endpoints `/api/v1/data/*`
- SQL direct : vues `v_current_*` pour Power BI
- Notebook : ce fichier pour l'exploration interactive